# Experiment 9 — Smoke factorial on degree=4 hardened substrate

**Sanity passed:** d=96 monolingual L_A on degree=4 landed at acc=0.554 (in-band,
lower edge). Now the two-language ρ × capacity factorial, smoke-sized.

## Design
- Grammar: locked from sanity — LEVELS=6, NT=4, T=4, degree=4, SEQ_LEN=512.
- Languages: L_A (anchor, identity order) + L_B (partial reversal, levels {4,5}).
  70/30 training mix. L_C deferred.
- ρ-conflict: 16 leaf-slots eligible (full density given grammar structure), each
  with a per-language alternative terminal. ρ=1.0 = no swap (control); ρ=0.5 = 50%
  swap rate at eligible positions.
- Data: large pool per language (12000 trees for L_B, 12000 for L_A). Small-data
  arm skipped at smoke.
- Capacity sweep: d ∈ {8, 32, 96}, k=4 fixed for geometric comparability.
- Seed: 0 only.
- Online sampling for anchor's held-out eval; **bounded pools for training**
  (so "data budget" is operationally defined when the full grid runs with small=1500).

## Measurements per (d, ρ)
- CE_L on held-out trees (never in any training pool), per language
- CE_B(ρ=0.5) - CE_B(ρ=1.0) = behavioral cost of conflict
- Chang-style mean-centered principal angle at mid layer, L_A vs L_B, within model
- Expected pattern: cost shrinks with d (sharing wins at low d) above some N* OR
  grows with d (separation wins when the model can afford it)

6 shared models × ~2.5 min each = ~15 min on H200.
Full grid commented in cell 1.

In [1]:
import os, time, math, random, statistics as _st
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
from contextlib import nullcontext

GPU_ID = 1
DEVICE = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")
print(f"using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  {torch.cuda.get_device_name(GPU_ID)}")

def autocast_ctx():
    return (torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16)
            if DEVICE.type == "cuda" else nullcontext())

torch.manual_seed(0)
random.seed(0)

using device: cuda:1
  NVIDIA H200 NVL


## 1. Config (grammar locked from sanity + factorial grid)

In [2]:
# Grammar (locked from sanity)
LEVELS       = 6
NT_SIZE      = 4
TERM_SIZE    = 4
MIN_DEGREE   = 4
MAX_DEGREE   = 4
RULE_LEN_CHOICES = [2, 3]

LANGS = {
    "L_A": set(),                       # anchor: identity
    "L_B": {4, 5},                      # partial reversal at levels 4,5
    "L_C": set(range(1, LEVELS + 1)),   # full reversal (unused this run)
}
LANG_NAMES = list(LANGS.keys())
ANCHOR = "L_A"
NONANCHOR = ["L_B"]                     # smoke: 1 non-anchor. Full: ["L_B","L_C"]

BOS_OFFSET = TERM_SIZE
BOS = {name: BOS_OFFSET + i for i, name in enumerate(LANG_NAMES)}
VOCAB_SIZE = TERM_SIZE + len(LANG_NAMES)

# ρ-conflict map density (max = NT_SIZE * TERM_SIZE, all leaf pairs eligible)
N_CONFLICT_SLOTS = NT_SIZE * TERM_SIZE   # 16

# ----- SMOKE grid -----
RHO_SWEEP    = [1.00, 0.50]            # full: [1.00, 0.85, 0.70, 0.50]
D_LIST       = [8, 32, 96]             # full: [8, 16, 32, 96]
SEEDS        = [0]                     # full: [0, 7]
DATA_SMALL   = 1500                    # trees per non-anchor language (unused in smoke)
DATA_LARGE   = 12000                   # trees per non-anchor language
ANCHOR_TREES = 12000                   # anchor always large
USE_SMALL    = False                   # smoke: skip small-data arm

MIX = {ANCHOR: 0.70, "L_B": 0.30}

# Training
N_HEAD      = 4
N_LAYER     = 8
SEQ_LEN     = 512
BATCH_SIZE  = 32
LR          = 3e-3
STEP_BUDGET = 8000

# Measurement
PATCH_LAYER = 5                        # 0-indexed; mid of 8 layers
K_FIXED     = 4                        # principal-angle k, fixed so d=8 is comparable
N_EVAL      = 300                      # held-out trees for CE/geometric eval

for d in D_LIST:
    assert d % N_HEAD == 0 and (d // N_HEAD) % 2 == 0

n_cells = len(RHO_SWEEP) * len(D_LIST) * len(SEEDS) * (2 if USE_SMALL else 1)
print(f"grammar: deg={MIN_DEGREE}  NT={NT_SIZE}  T={TERM_SIZE}  seq_len={SEQ_LEN}")
print(f"grid: {len(RHO_SWEEP)} ρ × {len(D_LIST)} d × {len(SEEDS)} seeds "
      f"× {'2 data' if USE_SMALL else 'large only'} = {n_cells} shared models")
print(f"mix: {MIX}  measurement layer: {PATCH_LAYER}  k_fixed: {K_FIXED}")
print(f"estimated: {n_cells * 2.5:.0f} min train + eval")

grammar: deg=4  NT=4  T=4  seq_len=512
grid: 2 ρ × 3 d × 1 seeds × large only = 6 shared models
mix: {'L_A': 0.7, 'L_B': 0.3}  measurement layer: 5  k_fixed: 4
estimated: 15 min train + eval


## 2. Grammar, sampler, linearizer, conflict map

In [3]:
def build_grammar(seed):
    rng = random.Random(seed)
    symbols_by_level = {1: [(1, 0)]}
    for l in range(2, LEVELS + 1):
        symbols_by_level[l] = [(l, i) for i in range(NT_SIZE)]
    terminals = list(range(TERM_SIZE))
    rules = defaultdict(list)
    for l in range(1, LEVELS + 1):
        child_level = l + 1
        child_symbols = (terminals if child_level == LEVELS + 1
                         else symbols_by_level[child_level])
        for sym in symbols_by_level[l]:
            degree = rng.randint(MIN_DEGREE, MAX_DEGREE)
            seen = set(); tries = 0
            while len(rules[sym]) < degree and tries < 5000:
                tries += 1
                rule_len = rng.choice(RULE_LEN_CHOICES)
                rhs = tuple(rng.choice(child_symbols) for _ in range(rule_len))
                if rhs in seen: continue
                seen.add(rhs); rules[sym].append(rhs)
    return rules, symbols_by_level

def sample_tree(rules, rng, symbol=(1, 0), level=1):
    rhs = rng.choice(rules[symbol])
    children = []
    for child_sym in rhs:
        if level == LEVELS:
            children.append(child_sym)
        else:
            children.append(sample_tree(rules, rng, symbol=child_sym, level=level + 1))
    return (symbol, children)

def _nleaves(n):
    return 1 if isinstance(n, int) else sum(_nleaves(c) for c in n[1])

def build_conflict_map(rules, seed):
    """For each non-anchor language, (parent_NT, orig_term) -> alternative_term.
    Picks N_CONFLICT_SLOTS pairs (default: ALL leaf-level pairs). Anchor = identity."""
    rng = random.Random(seed + 101)
    leaf_parents = [(LEVELS, i) for i in range(NT_SIZE)]
    all_pairs = [(p, t) for p in leaf_parents for t in range(TERM_SIZE)]
    chosen = all_pairs[:N_CONFLICT_SLOTS]
    cmap = {lang: defaultdict(dict) for lang in LANG_NAMES}
    for (parent, orig) in chosen:
        for lang in NONANCHOR + ["L_C"]:   # also L_C in case we add it later
            alt = rng.choice([t for t in range(TERM_SIZE) if t != orig])
            cmap[lang][parent][orig] = alt
    return cmap

def linearize_with_conflict(node, reverse_levels, cmap_lang, rho, rng, parent=None):
    """Linearize with word-order (reverse_levels) and apply ρ-conflict at terminals.
    At an eligible (parent_NT, orig_term) slot, with prob (1-ρ) swap to the
    language's alternative terminal. ρ=1 ⇒ no swaps (control)."""
    if isinstance(node, int):
        term = node
        if parent in cmap_lang and term in cmap_lang[parent] and rng.random() > rho:
            term = cmap_lang[parent][term]
        return [term]
    (level, idx), children = node
    order = list(reversed(children)) if level in reverse_levels else children
    tokens = []
    for c in order:
        tokens.extend(linearize_with_conflict(c, reverse_levels, cmap_lang, rho, rng,
                                              parent=(level, idx)))
    return tokens

# Build the grammar and the conflict map
substrate_seed = 0
rules, _ = build_grammar(substrate_seed)
cmap     = build_conflict_map(rules, substrate_seed)
print(f"grammar built: {sum(len(r) for r in rules.values())} rules across "
      f"{len(rules)} NTs")
n_slots_per_lang = sum(len(d) for d in cmap["L_B"].values())
print(f"conflict map: {n_slots_per_lang} (parent, orig_term) pairs per non-anchor lang")

grammar built: 84 rules across 21 NTs
conflict map: 16 (parent, orig_term) pairs per non-anchor lang


## 3. Pool builder (bounded training) + eval pack (held-out, matched trees)

In [4]:
def build_pool(rules, lang, cmap, rho, n_trees, seed):
    """Deterministic pool of (BOS + linearized tree) streams for one language under
    one ρ. 'Data budget' is operationalized here as n_trees. The ρ-conflict RNG is
    seeded, so the pool is reproducible — same tree, same conflict realization."""
    tree_rng = random.Random(seed)
    conf_rng = random.Random(seed + 7777)
    reverse_levels = LANGS[lang]
    cmap_lang = cmap[lang]
    stream = []
    for _ in range(n_trees):
        tree = sample_tree(rules, tree_rng)
        toks = linearize_with_conflict(tree, reverse_levels, cmap_lang, rho, conf_rng)
        stream.append(BOS[lang])
        stream.extend(toks)
    stream = torch.tensor(stream, dtype=torch.long)
    n_chunks = len(stream) // (SEQ_LEN + 1)
    assert n_chunks > 0, f"pool too small ({len(stream)} toks) for SEQ_LEN={SEQ_LEN}"
    packed = stream[: n_chunks * (SEQ_LEN + 1)].view(n_chunks, SEQ_LEN + 1)
    return packed

def build_eval_pack(rules, lang, cmap, rho, n_eval, seed):
    """Held-out eval set: n_eval trees, each one row (padded), with per-position
    leaf_id tagging the canonical anchor-order DFS index of each terminal — lets
    us match leaves across languages. Trees sampled with a dedicated seed, never
    overlaps a training pool."""
    tree_rng = random.Random(seed + 999999)
    conf_rng = random.Random(seed + 888888)
    reverse_levels = LANGS[lang]
    cmap_lang = cmap[lang]
    rows, leaf_rows = [], []
    for _ in range(n_eval):
        tree = sample_tree(rules, tree_rng)
        # canonical leaf-id by anchor-order (identity) DFS
        idmap = {}
        _c = [0]
        def _idwalk(n, path):
            if isinstance(n, int):
                idmap[tuple(path)] = _c[0]; _c[0] += 1; return
            (_lv, _ix), ch = n
            for k, ck in enumerate(ch): _idwalk(ck, path + [k])
        _idwalk(tree, [])
        toks, ids = [], []
        def _emit(n, path):
            if isinstance(n, int):
                term = n
                toks.append(term); ids.append(idmap[tuple(path)]); return
            (lv, ix), ch = n
            order = list(range(len(ch)))
            if lv in reverse_levels: order = list(reversed(order))
            for k in order:
                child = ch[k]
                if isinstance(child, int):
                    term = child
                    if (lv, ix) in cmap_lang and term in cmap_lang[(lv, ix)] \
                       and conf_rng.random() > rho:
                        term = cmap_lang[(lv, ix)][term]
                    toks.append(term); ids.append(idmap[tuple(path + [k])])
                else:
                    _emit(child, path + [k])
        _emit(tree, [])
        row = [BOS[lang]] + toks
        lids = [-1] + ids
        rows.append(row); leaf_rows.append(lids)
    maxlen = max(len(r) for r in rows)
    assert maxlen <= SEQ_LEN + 1, (
        f"eval tree of {maxlen} tokens exceeds context {SEQ_LEN+1}; raise SEQ_LEN")
    idx = torch.zeros((len(rows), maxlen), dtype=torch.long)
    leaf = torch.full((len(rows), maxlen), -1, dtype=torch.long)
    for t, (r, l) in enumerate(zip(rows, leaf_rows)):
        idx[t, :len(r)] = torch.tensor(r)
        leaf[t, :len(l)] = torch.tensor(l)
    return {"idx": idx, "leaf_id": leaf}

print("pool / eval builders ready")

pool / eval builders ready


## 4. TinyRotaryGPT (same as sanity)

In [5]:
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary(q, k, cos, sin):
    return ((q * cos) + (rotate_half(q) * sin),
            (k * cos) + (rotate_half(k) * sin))

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=1024, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)
    def forward(self, seq_len):
        return self.cos[:, :, :seq_len, :], self.sin[:, :, :seq_len, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head; self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.rotary = RotaryEmbedding(self.head_dim, max_seq_len)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rotary(T)
        q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_seq_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.GELU(),
                                 nn.Linear(4*n_embd, n_embd))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyRotaryGPT(nn.Module):
    def __init__(self, vocab_size, n_embd,
                 n_layer=N_LAYER, n_head=N_HEAD, max_seq_len=SEQ_LEN + 1):
        super().__init__()
        assert n_embd % n_head == 0 and (n_embd // n_head) % 2 == 0
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList(
            [Block(n_embd, n_head, max_seq_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for blk in self.blocks: x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                                   targets.reshape(-1))
        return logits, loss

def count_params(d):
    m = TinyRotaryGPT(VOCAB_SIZE, n_embd=d)
    return sum(p.numel() for p in m.parameters())

print(f"TinyRotaryGPT ready. params(d=96)={count_params(96):,}")

TinyRotaryGPT ready. params(d=96)=896,256


## 5. Train shared model on language mix (bounded pools)

In [6]:
def train_shared(pools_by_lang, mix, d_model, step_budget, seed):
    """Train one shared model on a language mixture, sampling language per batch
    by mix prob. Fixed step budget (convergence-matched). Returns final model."""
    torch.manual_seed(seed)
    model = TinyRotaryGPT(VOCAB_SIZE, n_embd=d_model).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    model.train()
    langs = list(pools_by_lang.keys())
    probs = torch.tensor([mix[l] for l in langs])
    data = {l: pools_by_lang[l].to(DEVICE) for l in langs}
    losses = []
    t0 = time.time()
    for step in range(step_budget):
        li = torch.multinomial(probs, 1).item()
        lang = langs[li]
        n = data[lang].shape[0]
        ridx = torch.randint(0, n, (BATCH_SIZE,), device=DEVICE)
        batch = data[lang][ridx]
        x, y = batch[:, :-1], batch[:, 1:]
        with autocast_ctx():
            _, loss = model(x, y)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    return model, losses, time.time() - t0

## 6. Eval — held-out CE per lang, principal-angle geometry L_A vs L_B

In [7]:
@torch.no_grad()
def eval_ce(model, eval_pack, max_rows=None):
    model.eval()
    idx = eval_pack["idx"].to(DEVICE)
    if max_rows: idx = idx[:max_rows]
    total_loss, total_toks = 0.0, 0
    for i in range(0, idx.shape[0], BATCH_SIZE):
        batch = idx[i:i+BATCH_SIZE]
        x, y = batch[:, :-1], batch[:, 1:]
        mask = (y != 0)                 # ignore padding
        with autocast_ctx():
            logits, _ = model(x, y)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                               y.reshape(-1), reduction="none").view_as(y)
        total_loss += (loss * mask).sum().item()
        total_toks += mask.sum().item()
    return total_loss / max(total_toks, 1)

@torch.no_grad()
def extract_mid_hiddens(model, idx_tensor, layer=PATCH_LAYER):
    model.eval()
    saved = []
    def hook(mod, inp, out): saved.append(out.detach().float().cpu())
    h = model.blocks[layer].register_forward_hook(hook)
    try:
        for i in range(0, idx_tensor.shape[0], BATCH_SIZE):
            batch = idx_tensor[i:i+BATCH_SIZE].to(DEVICE)
            with autocast_ctx(): _ = model(batch)
    finally:
        h.remove()
    return torch.cat(saved, dim=0)   # (n_rows, seq_len, d)

def _positions_for_leaves(pack):
    leaf = pack["leaf_id"]
    n_trees = leaf.shape[0]
    max_lid = int(leaf.max().item()) + 1
    pos = -torch.ones(n_trees, max_lid, dtype=torch.long)
    t_idx, p_idx = (leaf >= 0).nonzero(as_tuple=True)
    pos[t_idx, leaf[t_idx, p_idx]] = p_idx
    return pos

def collect_matched_activations(model, pack_A, pack_B, layer=PATCH_LAYER):
    """At matched leaves (same canonical leaf_id), gather hidden states from both
    languages. Returns two tensors of shape (n_matched, d)."""
    H_A = extract_mid_hiddens(model, pack_A["idx"], layer)
    H_B = extract_mid_hiddens(model, pack_B["idx"], layer)
    pos_A = _positions_for_leaves(pack_A)
    pos_B = _positions_for_leaves(pack_B)
    n_trees = min(pos_A.shape[0], pos_B.shape[0])
    max_lid = min(pos_A.shape[1], pos_B.shape[1])
    pos_A = pos_A[:n_trees, :max_lid]
    pos_B = pos_B[:n_trees, :max_lid]
    valid = (pos_A >= 0) & (pos_B >= 0)
    oa, ob = [], []
    for t in range(n_trees):
        lids = valid[t].nonzero(as_tuple=True)[0]
        if lids.numel() == 0: continue
        oa.append(H_A[t, pos_A[t, lids]])
        ob.append(H_B[t, pos_B[t, lids]])
    return torch.cat(oa, 0), torch.cat(ob, 0)

def principal_angles_mean(X, Y, k):
    """Mean-centered, fixed-k principal angles (Chang-style). Returns mean angle (deg)."""
    Xc = X - X.mean(dim=0, keepdim=True)
    Yc = Y - Y.mean(dim=0, keepdim=True)
    Ux, _, _ = torch.linalg.svd(Xc, full_matrices=False)
    Uy, _, _ = torch.linalg.svd(Yc, full_matrices=False)
    k = min(k, Ux.shape[1], Uy.shape[1])
    M = Ux[:, :k].T @ Uy[:, :k]
    s = torch.linalg.svdvals(M).clamp(-1.0, 1.0)
    angles_deg = torch.rad2deg(torch.arccos(s))
    return angles_deg.mean().item()

print("eval + geometry ready")

eval + geometry ready


## 7. Factorial sweep

In [8]:
results = []
total_t0 = time.time()

for seed in SEEDS:
    data_arms = ([("large", DATA_LARGE)]
                 + ([("small", DATA_SMALL)] if USE_SMALL else []))
    for data_name, n_trees_B in data_arms:
        for d in D_LIST:
            # Pre-build eval packs once per (seed, data) — same held-out set across rho/d
            # (eval pack ρ: use ρ for L_B so eval content matches training distribution)
            for rho in RHO_SWEEP:
                print(f"\n[cell] seed={seed} data={data_name} d={d} ρ={rho:.2f}")

                # Build pools fresh per cell (ρ enters here)
                pool_A = build_pool(rules, ANCHOR, cmap, 1.00, ANCHOR_TREES, seed)
                pool_B = build_pool(rules, "L_B", cmap, rho, n_trees_B, seed + 1)
                pools = {ANCHOR: pool_A, "L_B": pool_B}
                print(f"  pools: A={pool_A.shape[0]} chunks ({ANCHOR_TREES} trees), "
                      f"B={pool_B.shape[0]} chunks ({n_trees_B} trees)")

                # Train one shared model
                model, losses, train_s = train_shared(pools, MIX, d, STEP_BUDGET, seed)
                final_loss = _st.mean(losses[-100:])
                print(f"  train: {train_s/60:.1f} min  final tail-100 loss={final_loss:.4f}")

                # Held-out eval packs — fresh trees, ρ matches training condition
                ev_A = build_eval_pack(rules, ANCHOR, cmap, 1.00, N_EVAL, seed + 42)
                ev_B = build_eval_pack(rules, "L_B", cmap, rho, N_EVAL, seed + 43)

                ce_A = eval_ce(model, ev_A)
                ce_B = eval_ce(model, ev_B)

                # Geometry: within-model L_A vs L_B at matched leaves, mid layer
                X_A, X_B = collect_matched_activations(model, ev_A, ev_B)
                mean_ang = principal_angles_mean(X_A, X_B, K_FIXED)

                print(f"  CE_A={ce_A:.4f}  CE_B={ce_B:.4f}  "
                      f"mean_angle(L_A,L_B)={mean_ang:.2f}°")

                results.append(dict(
                    seed=seed, data=data_name, d=d, rho=rho,
                    ce_A=ce_A, ce_B=ce_B, mean_ang=mean_ang,
                    final_loss=final_loss, train_min=train_s/60,
                ))

                # Free before next cell
                del model
                if DEVICE.type == "cuda": torch.cuda.empty_cache()

print(f"\n[sweep done in {(time.time()-total_t0)/60:.1f} min total]")


[cell] seed=0 data=large d=8 ρ=1.00
  pools: A=6308 chunks (12000 trees), B=6296 chunks (12000 trees)
  train: 1.2 min  final tail-100 loss=1.0666
  CE_A=1.1471  CE_B=1.1519  mean_angle(L_A,L_B)=88.93°

[cell] seed=0 data=large d=8 ρ=0.50
  pools: A=6308 chunks (12000 trees), B=6296 chunks (12000 trees)
  train: 1.3 min  final tail-100 loss=1.1869
  CE_A=1.1493  CE_B=1.3445  mean_angle(L_A,L_B)=87.89°

[cell] seed=0 data=large d=32 ρ=1.00
  pools: A=6308 chunks (12000 trees), B=6296 chunks (12000 trees)
  train: 1.1 min  final tail-100 loss=0.9929
  CE_A=1.0569  CE_B=1.0497  mean_angle(L_A,L_B)=89.67°

[cell] seed=0 data=large d=32 ρ=0.50
  pools: A=6308 chunks (12000 trees), B=6296 chunks (12000 trees)
  train: 0.9 min  final tail-100 loss=1.1005
  CE_A=1.0655  CE_B=1.3099  mean_angle(L_A,L_B)=89.80°

[cell] seed=0 data=large d=96 ρ=1.00
  pools: A=6308 chunks (12000 trees), B=6296 chunks (12000 trees)
  train: 1.1 min  final tail-100 loss=0.9775
  CE_A=1.0474  CE_B=1.0473  mean_angl

## 8. Summary table + ΔCE / Δangle vs capacity

In [9]:
print("=" * 78)
print(f"FACTORIAL RESULTS (deg={MIN_DEGREE}, layer={PATCH_LAYER}, k={K_FIXED})")
print("=" * 78)
hdr = f"{'seed':>4} {'data':>6} {'d':>4} {'ρ':>5} {'CE_A':>7} {'CE_B':>7} {'angle':>7}"
print(hdr); print("-" * 78)
for r in results:
    print(f"{r['seed']:>4} {r['data']:>6} {r['d']:>4} {r['rho']:>5.2f} "
          f"{r['ce_A']:>7.4f} {r['ce_B']:>7.4f} {r['mean_ang']:>6.2f}°")

# ΔCE_B and Δangle = (ρ=0.5 minus ρ=1.0) per (seed, data, d)
# This is the behavioral + geometric cost of conflict, which should either
# shrink with d (sharing wins at low d above N*) or climb with d (separation
# wins when the model can afford it). Direction + magnitude = the main signal.
print("\n" + "=" * 78)
print("GAPS: ρ=0.5 − ρ=1.0 per (seed, data, d)")
print("=" * 78)
print(f"{'seed':>4} {'data':>6} {'d':>4} {'ΔCE_B':>8} {'Δangle':>8}")
print("-" * 78)
by_cell = {}
for r in results:
    key = (r['seed'], r['data'], r['d'])
    by_cell.setdefault(key, {})[r['rho']] = r

for (seed, data, d), rhos in sorted(by_cell.items()):
    if 1.00 in rhos and 0.50 in rhos:
        d_ce = rhos[0.50]['ce_B'] - rhos[1.00]['ce_B']
        d_ang = rhos[0.50]['mean_ang'] - rhos[1.00]['mean_ang']
        print(f"{seed:>4} {data:>6} {d:>4} {d_ce:>+8.4f} {d_ang:>+7.2f}°")

print("\nREADING THE TABLE:")
print("  ΔCE_B > 0 everywhere: conflict costs accuracy (expected).")
print("  ΔCE_B shrinks with d: sharing wins at low d, separation becomes cheap at high d.")
print("    (= N* story — the capacity-crossover prediction.)")
print("  Δangle grows with d: representations diverge more when model can afford it.")
print("    (= geometric correlate of the same story.)")
print("  Both flat across d: no capacity gating — need wider d or different mechanism.")
print("  ΔCE_B ≈ 0 everywhere: conflict is too cheap at this density — raise mechanism load.")

FACTORIAL RESULTS (deg=4, layer=5, k=4)
seed   data    d     ρ    CE_A    CE_B   angle
------------------------------------------------------------------------------
   0  large    8  1.00  1.1471  1.1519  88.93°
   0  large    8  0.50  1.1493  1.3445  87.89°
   0  large   32  1.00  1.0569  1.0497  89.67°
   0  large   32  0.50  1.0655  1.3099  89.80°
   0  large   96  1.00  1.0474  1.0473  89.53°
   0  large   96  0.50  1.0631  1.3255  89.76°

GAPS: ρ=0.5 − ρ=1.0 per (seed, data, d)
seed   data    d    ΔCE_B   Δangle
------------------------------------------------------------------------------
   0  large    8  +0.1926   -1.04°
   0  large   32  +0.2602   +0.13°
   0  large   96  +0.2782   +0.23°

READING THE TABLE:
  ΔCE_B > 0 everywhere: conflict costs accuracy (expected).
  ΔCE_B shrinks with d: sharing wins at low d, separation becomes cheap at high d.
    (= N* story — the capacity-crossover prediction.)
  Δangle grows with d: representations diverge more when model can afford i